# D-MeZO — Bootstrap notebook for Google Colab Pro+

Target hardware: **RTX PRO 6000 Blackwell (96 GB)** или A100 80 GB.

Этот ноутбук:
1. Монтирует Google Drive (для чекпоинтов).
2. Клонирует или копирует проект `dmezo`.
3. Устанавливает зависимости.
4. Проверяет GPU.
5. Запускает Day 1 sanity check (MeZO на Qwen3-4B / SST-2).

**Бюджет:** ~30-50 compute units на полный прогон Day 1.

## 0. Mount Google Drive для чекпоинтов

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUNS_DIR = '/content/drive/MyDrive/dmezo_runs'
os.makedirs(RUNS_DIR, exist_ok=True)
print(f'Runs will be saved to: {RUNS_DIR}')

## 1. Клонировать проект из GitHub

Установи `GH_REPO` ниже на свой приватный repo (e.g. `your-username/dmezo`). Если repo приватный, добавь Personal Access Token через Colab Secrets (key icon в sidebar): создай secret `GH_TOKEN` с `repo` scope, в нём — твой PAT.

In [ ]:
GH_REPO = 'Siesher/dmezo'  # private repo on github.com

import os, shutil
if os.path.exists('/content/dmezo'):
    shutil.rmtree('/content/dmezo')

# Try to read PAT from Colab Secrets (for private repos).
try:
    from google.colab import userdata
    gh_token = userdata.get('GH_TOKEN')
    clone_url = f'https://{gh_token}@github.com/{GH_REPO}.git'
except Exception:
    clone_url = f'https://github.com/{GH_REPO}.git'  # works only if repo is public

!git clone {clone_url} /content/dmezo
%cd /content/dmezo
!git log --oneline -5

## 2. Установить зависимости

In [ ]:
!pip install -q --upgrade pip
!pip install -q -e .
# Flash attention (опционально, ускоряет MeZO forward на Blackwell):
!pip install -q flash-attn --no-build-isolation || echo 'flash-attn install skipped'

## 3. Проверить GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    total_gb = torch.cuda.mem_get_info()[1] / 1e9
    print(f'GPU: {name}')
    print(f'Memory: {free_gb:.1f} / {total_gb:.1f} GB free')
    print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')
    cc = torch.cuda.get_device_capability(0)
    print(f'Compute capability: {cc} (Blackwell = (10, 0) or higher)')

## 4. Запустить тесты

Должны пройти все: perturbation determinism + topology properties.

In [ ]:
!pytest tests/ -v

## 5. Day 1 sanity check — MeZO на Qwen3-4B / SST-2

Скрипт пишет JSONL-лог и чекпоинты в `experiments/01_sanity_qwen3_4b_sst2/` (внутри `/content/dmezo`). Чтобы сохранить в Drive, передадим `--output-dir`.

In [ ]:
OUTPUT_DIR = f'{RUNS_DIR}/01_sanity_qwen3_4b_sst2'
MLFLOW_URI = f'file:{RUNS_DIR}/mlruns'
!python scripts/01_sanity_check_mezo.py \
    --config configs/qwen3_4b_sst2.yaml \
    --output-dir {OUTPUT_DIR} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-4b-sst2-day1

## 6. Анализ результата

## 6a. MLflow run summary

MLflow data persists in `{RUNS_DIR}/mlruns/` (на Drive). Можно посмотреть локально через `mlflow ui --backend-store-uri file:./mlruns` после скачивания, или прямо здесь в ноутбуке:

In [ ]:
import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)
exp = mlflow.get_experiment_by_name('dmezo_sanity')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=['attribute.start_time DESC'])
display(runs[['run_id', 'tags.mlflow.runName', 'tags.sanity_verdict',
              'metrics.init_eval_loss', 'metrics.final_eval_loss', 'metrics.eval_loss_drop_pct',
              'params.model.name', 'params.mezo.lr', 'params.mezo.eps']].head(10))

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

log_path = f'{OUTPUT_DIR}/log.jsonl'
rows = [json.loads(line) for line in open(log_path)]
df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train = df[df['train_loss'].notna()] if 'train_loss' in df.columns else df.iloc[0:0]
ev = df[df.get('eval_loss').notna()] if 'eval_loss' in df.columns else df.iloc[0:0]
if not train.empty:
    axes[0].plot(train['step'], train['train_loss'], label='train (moving avg)')
if not ev.empty:
    axes[0].plot(ev['step'], ev['eval_loss'], 'o-', label='eval')
axes[0].set_xlabel('MeZO step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Day 1 sanity: MeZO on Qwen3-4B / SST-2')
axes[0].legend()
axes[0].grid(alpha=0.3)

if 'projected_grad' in df.columns:
    pg = df[df['projected_grad'].notna()]
    axes[1].plot(pg['step'], pg['projected_grad'])
    axes[1].set_xlabel('MeZO step')
    axes[1].set_ylabel('Projected gradient')
    axes[1].set_title('Projected gradient (should be non-vanishing, non-NaN)')
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sanity_plot.png', dpi=120)
plt.show()
print('Saved plot to', f'{OUTPUT_DIR}/sanity_plot.png')

## 7. Что дальше

Если sanity прошёл (eval loss упал на >10%):
- Переходить к Day 2 чтению FedKSeed/FedZeN.
- Запустить centralized baselines на BoolQ/COPA фоном.

Если sanity провалился:
- Проверить hyperparameters: `lr` в диапазоне 1e-7 - 1e-5, `eps=1e-3`.
- Проверить, что `param.requires_grad=True` для всех параметров.
- Распечатать `model.config` — убедиться, что Qwen3-4B загрузился именно как ожидалось.
- Запустить debug-версию с 10 steps и руками посмотреть на projected_grad.

## 8. Bonus: Qwen3.5-4B-Base — hybrid linear-attention sanity

Qwen3.5-4B-Base — это **vision-language модель с hybrid linear-attention text decoder** (32 слоя: 24 linear + 8 full attention, 3:1 ratio). Loader заморозит vision tower; MeZO будет perturbать только text decoder (~4.4B trainable params).

**Зачем:** Princeton MeZO 2023 проверил только на full-attention моделях. Это первый известный нам test MeZO на hybrid linear-attention арх. Любой результат публикабелен.

Если уже клонировал proj и хочешь подтянуть свежий код — сделай `!cd /content/dmezo && git pull` перед запуском ячейки ниже.

In [ ]:
OUTPUT_DIR_Q35 = f'{RUNS_DIR}/01_sanity_qwen3_5_4b_base_sst2'
!python scripts/01_sanity_check_mezo.py \
    --config configs/qwen3_5_4b_base_sst2.yaml \
    --output-dir {OUTPUT_DIR_Q35} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-5-4b-base-sst2-hybrid-arch

In [ ]:
# Side-by-side: Qwen3-4B (Day 1) vs Qwen3.5-4B-Base (hybrid)
import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)
exp = mlflow.get_experiment_by_name('dmezo_sanity')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=['attribute.start_time DESC'])
cols = ['tags.mlflow.runName', 'params.model.name', 'tags.sanity_verdict',
        'metrics.init_eval_loss', 'metrics.final_eval_loss', 'metrics.eval_loss_drop_pct']
display(runs[cols].head(10))

## 9. Cross-dataset check — BoolQ on both architectures

После SST-2 Day 1 (Qwen3-4B PASS 88%) и Section 8 (Qwen3.5-4B-Base hybrid PASS 94.7%) — закрепляем finding на **BoolQ** (yes/no QA на длинных passages, SuperGLUE). Если оба тоже PASS — у нас 2×2 design (architecture × task), который перекрывает основной paper-claim risk "это специфика SST-2".

Сначала `!cd /content/dmezo && git pull` если notebook уже клонирован, иначе перезапусти runtime и re-run cells 1-6.

In [ ]:
!cd /content/dmezo && git pull

# Qwen3-4B / BoolQ (full attention baseline)
OUTPUT_DIR_Q3_BQ = f'{RUNS_DIR}/02_sanity_qwen3_4b_boolq'
!python scripts/01_sanity_check_mezo.py \
    --config configs/qwen3_4b_boolq.yaml \
    --output-dir {OUTPUT_DIR_Q3_BQ} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-4b-boolq

In [ ]:
# Qwen3.5-4B-Base / BoolQ (hybrid linear-attention)
OUTPUT_DIR_Q35_BQ = f'{RUNS_DIR}/02_sanity_qwen3_5_4b_base_boolq'
!python scripts/01_sanity_check_mezo.py \
    --config configs/qwen3_5_4b_base_boolq.yaml \
    --output-dir {OUTPUT_DIR_Q35_BQ} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-5-4b-base-boolq-hybrid

In [ ]:
# 2x2 design summary: architecture x task
import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)
exp = mlflow.get_experiment_by_name('dmezo_sanity')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=['attribute.start_time DESC'])
cols = ['tags.mlflow.runName', 'params.model.name', 'params.data.task',
        'tags.sanity_verdict', 'metrics.init_eval_loss', 'metrics.final_eval_loss',
        'metrics.eval_loss_drop_pct']
display(runs[cols].head(10))